# 🛰️ Nova Neural Fusion: 500+ Artists Massive Training
Este cuaderno carga un dataset externo de 500+ artistas para un entrenamiento profundo.

### 🚀 Instrucciones:
1. Sube el archivo `art_universe_500.jsonl` a la carpeta de archivos de Colab (icono de carpeta a la izquierda).
2. Ejecuta todo.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3.2-1b-instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# CARGA MASIVA DEL DATASET DE 500 ARTISTAS
dataset = load_dataset("json", data_files="art_universe_500.jsonl", split="train")

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = f"### Instruction:\n{instruction}\n\n### Response:\n{output}"
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True,)

model = FastLanguageModel.get_peft_model(model, r = 16, lora_alpha = 16, random_state = 3407)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        max_steps = 300, # Aumentado para absorber los 500 artistas
        learning_rate = 2e-4,
        output_dir = "outputs",
        logging_steps = 1
    ),
)
trainer.train()

model.save_pretrained_gguf("nova_ultra_500_artists", tokenizer, quantization_method = "q4_k_m")
from google.colab import files
files.download("nova_ultra_500_artists-unsloth.Q4_K_M.gguf")